# 04 · Direct attribution: who writes the answer?

The final residual stream is a **sum**: embedding + 48 head outputs + 8 MLP outputs + biases. The output head is linear. The final LayerNorm is not, but freezing its per-token scale at its actual value makes the split linear, and exact for this input. So every component's **direct** contribution to a field's tendency can be drawn as a map, and the maps add up to the model's output.

*Direct* means what a component writes straight into the output, not what it does through later layers. Notebook 03's ablation measures the *total* effect, and comparing the two is informative.

In [ ]:
%matplotlib inline
import sys, os
from pathlib import Path
try:
    import vit_interp
except ImportError:  # not installed: use this checkout, $VIT_INTERP_REPO, or the shared one
    for repo in (Path.cwd().parent, Path(os.environ.get("VIT_INTERP_REPO", "/home/ac.ngmahfouz/vit-interp"))):
        if (repo / "vit_interp").is_dir():
            sys.path.insert(0, str(repo)); break
import numpy as np, torch, matplotlib.pyplot as plt
from vit_interp import playground as pg
em, meta = pg.load()      # or pg.load("/path/to/checkpoint.pt"); $PLAYGROUND_CKPT also works
FAST = pg.FAST            # PLAYGROUND_FAST=1 -> small settings (CPU tests)

In [ ]:
x = pg.state("2010-01-15T00")
mask = pg.box(35, 60, 280, 320)            # North Atlantic
maps, table = pg.direct_attribution(em, x, "V_3", mask)
print("total in box:", round(table.attrs["total"], 3), " reconstruction error:", table.attrs["max_abs_reconstruction_error"])
table.head(10).round(3)

In [ ]:
top = table.component.head(6).tolist()
pg.plot_maps({f"{c}: {table.set_index('component').contribution[c]:+.2f}": maps[c] for c in top}, cols=3, shared=True);

Sum per layer: are heads or MLPs doing most of the writing, and at which depth?

In [ ]:
t = table.set_index("component").contribution
layers = range(em.cfg.depth)
heads = [sum(t[f"L{i}H{h}"] for h in range(em.cfg.heads)) for i in layers]
mlps = [t[f"L{i}MLP"] for i in layers]
fig, ax = plt.subplots(figsize=(7, 3)); ax.bar(np.array(list(layers)) - 0.2, heads, 0.4, label="heads")
ax.bar(np.array(list(layers)) + 0.2, mlps, 0.4, label="MLP"); ax.axhline(0, color="k", lw=0.5); ax.legend(); ax.set_xlabel("layer");

Direct (this notebook) vs total (ablation) effect, for the same components:

In [ ]:
names = table.component.head(10 if FAST else 20).tolist()
names = [n for n in names if n not in ("embed", "biases")]
metric = lambda tt: pg.box_mean(tt[0, pg.ch("V_3")], mask)
sw = pg.sweep_components(em, x, metric, which=names, mode="zero").set_index("component")
fig, ax = plt.subplots(figsize=(4.5, 4))
d = table.set_index("component").contribution[names]; tot = -sw.effect[names]
ax.scatter(d, tot); [ax.annotate(n, (d[n], tot[n]), fontsize=7) for n in names]
ax.axline((0, 0), slope=1, color="k", lw=0.5); ax.set_xlabel("direct contribution"); ax.set_ylabel("total effect (−Δ under zero ablation)");

## Try this

1. Move the box to a **developing cyclone** or a **trough** (pick one from the V_3 tendency map in notebook 00). Do the same components show up?
2. Attribute **T_7** or **specific_total_water_7** instead. Is moisture written by MLPs (local physics) and winds by heads (transport)?
3. Off-diagonal points in the scatter are components that act **through other components**. Those are where circuits start. Take one, ablate it, and look at which later heads change their output (`em.step(..., cache=..., names={...})`).
4. The embedding term is the "persistence-like" linear map of each patch onto itself. How big is it compared with the heads?